In [12]:
# =========================
# Imports Projeto
# =========================

import sys
from pathlib import Path

# ------------------------------------------
# Sobe um nível a partir de jobs/
# para encontrar a pasta src
# ------------------------------------------

if str(Path.cwd().parent) not in sys.path:
    sys.path.append(str(Path.cwd().parent))

from src.config.secrets import get_secret

# =========================
# Container Bronze
# =========================

BRONZE_CONTAINER = get_secret(
    "AZURE-CONTAINER-BRONZE",
    "AZURE_CONTAINER_BRONZE"
)

# =========================
# Validação
# =========================

if not BRONZE_CONTAINER:
    raise ValueError(
        "❌ Container Bronze não configurado."
    )

# =========================
# Log
# =========================

print(
    f"✅ Container Bronze: "
    f"{BRONZE_CONTAINER}"
)

✅ Container Bronze: bronze


In [13]:
# =========================
# Imports Projeto
# =========================

import sys
from pathlib import Path

# sobe um nível a partir de jobs/

if str(Path.cwd().parent) not in sys.path:
    sys.path.append(str(Path.cwd().parent))

from src.config.secrets import get_secret

# =========================
# Imports
# =========================

from azure.storage.blob import BlobServiceClient

# =========================
# Azure Blob
# =========================

try:

    # ------------------------------------------
    # Azure Storage
    # ------------------------------------------

    AZURE_STORAGE_ACCOUNT = get_secret(
        "AZURE-STORAGE-ACCOUNT",
        "AZURE_STORAGE_ACCOUNT"
    )

    AZURE_STORAGE_KEY = get_secret(
        "AZURE-STORAGE-KEY",
        "AZURE_STORAGE_KEY"
    )

    # ------------------------------------------
    # Validações
    # ------------------------------------------

    if not AZURE_STORAGE_ACCOUNT:
        raise ValueError(
            "❌ AZURE-STORAGE-ACCOUNT não configurada."
        )

    if not AZURE_STORAGE_KEY:
        raise ValueError(
            "❌ AZURE-STORAGE-KEY não configurada."
        )

    # ------------------------------------------
    # URL Storage Account
    # ------------------------------------------

    account_url = (
        f"https://{AZURE_STORAGE_ACCOUNT}.blob.core.windows.net"
    )

    # ------------------------------------------
    # Cliente Blob Storage
    # ------------------------------------------

    blob_service_client = BlobServiceClient(
        account_url=account_url,
        credential=AZURE_STORAGE_KEY
    )

    # ------------------------------------------
    # Logs
    # ------------------------------------------

    print(
        "✅ Cliente Azure Blob inicializado."
    )

    print(
        f"✅ Storage Account: "
        f"{AZURE_STORAGE_ACCOUNT}"
    )

    print(
        f"✅ Account URL: "
        f"{account_url}"
    )

except Exception as e:

    print(
        f"❌ Erro Azure Blob: {e}"
    )

    raise

✅ Cliente Azure Blob inicializado.
✅ Storage Account: stfiapoin4ci2kb4w7c
✅ Account URL: https://stfiapoin4ci2kb4w7c.blob.core.windows.net


In [14]:
# =========================
# API IBGE
# =========================

import requests

# ------------------------------------------
# Endpoint IBGE
# ------------------------------------------

url = (
    "https://servicodados.ibge.gov.br/"
    "api/v1/localidades/municipios"
)

# ------------------------------------------
# Requisição
# ------------------------------------------

response = requests.get(
    url,
    timeout=60
)

response.raise_for_status()

# ------------------------------------------
# JSON
# ------------------------------------------

data = response.json()

# ------------------------------------------
# Validação
# ------------------------------------------

if not data:
    raise ValueError("❌ Nenhum município retornado pela API do IBGE." )

# ------------------------------------------
# Logs
# ------------------------------------------

print(
    f"✅ Municípios encontrados: "
    f"{len(data)}"
)


✅ Municípios encontrados: 5571


In [15]:
# =========================
# DataFrame
# =========================

import pandas as pd
import datetime as dt

# ------------------------------------------
# Validação API
# ------------------------------------------

if not isinstance(data, list):
    raise ValueError(
        f"❌ Tipo inesperado retornado pela API: {type(data)}"
    )

if len(data) == 0:
    raise ValueError(
        "❌ API retornou lista vazia."
    )

# ------------------------------------------
# DataFrame
# ------------------------------------------

df = pd.json_normalize(data)

# ------------------------------------------
# Renomeia colunas
# ------------------------------------------

df = df.rename(
    columns={
        "id": "municipio_id",
        "nome": "municipio_nome",
        "microrregiao.id": "microrregiao_id",
        "microrregiao.nome": "microrregiao_nome",
        "microrregiao.mesorregiao.id": "mesorregiao_id",
        "microrregiao.mesorregiao.nome": "mesorregiao_nome",
        "microrregiao.mesorregiao.UF.id": "estado_id",
        "microrregiao.mesorregiao.UF.sigla": "estado_sigla",
        "microrregiao.mesorregiao.UF.nome": "estado_nome",
        "microrregiao.mesorregiao.UF.regiao.id": "regiao_id",
        "microrregiao.mesorregiao.UF.regiao.sigla": "regiao_sigla",
        "microrregiao.mesorregiao.UF.regiao.nome": "regiao_nome",
        "regiao-imediata.id": "regiao_imediata_id",
        "regiao-imediata.nome": "regiao_imediata_nome",
        "regiao-imediata.regiao-intermediaria.id": "regiao_intermediaria_id",
        "regiao-imediata.regiao-intermediaria.nome": "regiao_intermediaria_nome",
        "regiao-imediata.regiao-intermediaria.UF.id": "ri_estado_id",
        "regiao-imediata.regiao-intermediaria.UF.sigla": "ri_estado_sigla",
        "regiao-imediata.regiao-intermediaria.UF.nome": "ri_estado_nome",
        "regiao-imediata.regiao-intermediaria.UF.regiao.id": "ri_regiao_id",
        "regiao-imediata.regiao-intermediaria.UF.regiao.sigla": "ri_regiao_sigla",
        "regiao-imediata.regiao-intermediaria.UF.regiao.nome": "ri_regiao_nome"
    }
)

# ------------------------------------------
# Metadados de ingestão
# ------------------------------------------

df["_ingested_at"] = (
    dt.datetime.now(
        dt.timezone.utc
    ).isoformat()
)

# ------------------------------------------
# Debug
# ------------------------------------------

print(
    "✅ Colunas disponíveis:"
)

for column in sorted(df.columns):
    print(column)

# ------------------------------------------
# Ordenação
# ------------------------------------------

if (
    "estado_sigla" in df.columns
    and "municipio_nome" in df.columns
):

    df = (
        df
        .sort_values(
            by=[
                "estado_sigla",
                "municipio_nome"
            ]
        )
        .reset_index(
            drop=True
        )
    )

# ------------------------------------------
# Validação DataFrame
# ------------------------------------------

if df.empty:
    raise ValueError(
        "❌ DataFrame IBGE Municípios vazio."
    )

required_columns = [
    "municipio_id",
    "municipio_nome",
    "_ingested_at"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"❌ Colunas ausentes: {missing_columns}"
    )

# ------------------------------------------
# Tipos de dados
# ------------------------------------------

for column in [
    "municipio_id",
    "estado_id",
    "regiao_id",
    "microrregiao_id",
    "mesorregiao_id"
]:

    if column in df.columns:

        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        ).astype("Int64")

# ------------------------------------------
# Logs
# ------------------------------------------

print(
    f"✅ Municípios carregados: "
    f"{len(df)}"
)

print(
    f"✅ Colunas carregadas: "
    f"{len(df.columns)}"
)

if "estado_sigla" in df.columns:

    print(
        f"✅ Estados únicos: "
        f"{df['estado_sigla'].nunique()}"
    )

if "regiao_sigla" in df.columns:

    print(
        f"✅ Regiões únicas: "
        f"{df['regiao_sigla'].nunique()}"
    )

print(
    f"✅ Municípios únicos: "
    f"{df['municipio_id'].nunique()}"
)

print(
    f"✅ Timestamp de ingestão: "
    f"{df['_ingested_at'].iloc[0]}"
)

# ------------------------------------------
# Preview
# ------------------------------------------

display(df)

✅ Colunas disponíveis:
_ingested_at
estado_id
estado_nome
estado_sigla
mesorregiao_id
mesorregiao_nome
microrregiao
microrregiao_id
microrregiao_nome
municipio_id
municipio_nome
regiao_id
regiao_imediata_id
regiao_imediata_nome
regiao_intermediaria_id
regiao_intermediaria_nome
regiao_nome
regiao_sigla
ri_estado_id
ri_estado_nome
ri_estado_sigla
ri_regiao_id
ri_regiao_nome
ri_regiao_sigla
✅ Municípios carregados: 5571
✅ Colunas carregadas: 24
✅ Estados únicos: 27
✅ Regiões únicas: 5
✅ Municípios únicos: 5571
✅ Timestamp de ingestão: 2026-08-26T08:32:46.891092+00:00


,municipio_id,municipio_nome,microrregiao_id,microrregiao_nome,mesorregiao_id,mesorregiao_nome,estado_id,estado_sigla,estado_nome,regiao_id,...,regiao_intermediaria_id,regiao_intermediaria_nome,ri_estado_id,ri_estado_sigla,ri_estado_nome,ri_regiao_id,ri_regiao_sigla,ri_regiao_nome,microrregiao,_ingested_at
0,1200013,Acrelândia,12004,Rio Branco,1202,Vale do Acre,12,AC,Acre,1,...,1201,Rio Branco,12,AC,Acre,1,N,Norte,NaN,2026-08-26T08:32:46.891092+00:00
1,1200054,Assis Brasil,12005,Brasiléia,1202,Vale do Acre,12,AC,Acre,1,...,1201,Rio Branco,12,AC,Acre,1,N,Norte,NaN,2026-08-26T08:32:46.891092+00:00
2,1200104,Brasiléia,12005,Brasiléia,1202,Vale do Acre,12,AC,Acre,1,...,1201,Rio Branco,12,AC,Acre,1,N,Norte,NaN,2026-08-26T08:32:46.891092+00:00
3,1200138,Bujari,12004,Rio Branco,1202,Vale do Acre,12,AC,Acre,1,...,1201,Rio Branco,12,AC,Acre,1,N,Norte,NaN,2026-08-26T08:32:46.891092+00:00
4,1200179,Capixaba,12004,Rio Branco,1202,Vale do Acre,12,AC,Acre,1,...,1201,Rio Branco,12,AC,Acre,1,N,Norte,NaN,2026-08-26T08:32:46.891092+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5566,1721257,Tupirama,17003,Miracema do Tocantins,1701,Ocidental do Tocantins,17,TO,Tocantins,1,...,1702,Araguaína,17,TO,Tocantins,1,N,Norte,NaN,2026-08-26T08:32:46.891092+00:00
5567,1721307,Tupiratins,17003,Miracema do Tocantins,1701,Ocidental do Tocantins,17,TO,Tocantins,1,...,1702,Araguaína,17,TO,Tocantins,1,N,Norte,NaN,2026-08-26T08:32:46.891092+00:00
5568,1722081,Wanderlândia,17002,Araguaína,1701,Ocidental do Tocantins,17,TO,Tocantins,1,...,1702,Araguaína,17,TO,Tocantins,1,N,Norte,NaN,2026-08-26T08:32:46.891092+00:00
5569,1722107,Xambioá,17002,Araguaína,1701,Ocidental do Tocantins,17,TO,Tocantins,1,...,1702,Araguaína,17,TO,Tocantins,1,N,Norte,NaN,2026-08-26T08:32:46.891092+00:00


In [16]:
# =========================
# Salvar Parquet
# =========================

from pathlib import Path
import datetime as dt

# ------------------------------------------
# Validação DataFrame
# ------------------------------------------

if df.empty:
    raise ValueError(
        "❌ DataFrame vazio. Nada para salvar."
    )

# ------------------------------------------
# Diretório temporário
# ------------------------------------------

temp_dir = Path.cwd() / "tmp"

temp_dir.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------
# Nome do arquivo
# ------------------------------------------

date_suffix = dt.datetime.now().strftime(
    "%Y-%m-%d"
)

parquet_file = (
    temp_dir
    / f"{date_suffix}_ibge_municipios.parquet"
)

# ------------------------------------------
# Salva Parquet
# ------------------------------------------

df.to_parquet(
    parquet_file,
    index=False
)

# ------------------------------------------
# Valida criação
# ------------------------------------------

if not parquet_file.exists():
    raise FileNotFoundError(
        f"❌ Arquivo não foi criado: {parquet_file}"
    )

file_size_bytes = parquet_file.stat().st_size
file_size_kb = round(
    file_size_bytes / 1024,
    2
)

if file_size_bytes == 0:
    raise ValueError(
        f"❌ Arquivo criado sem conteúdo: {parquet_file}"
    )

# ------------------------------------------
# Logs
# ------------------------------------------

print(
    f"✅ Parquet criado: "
    f"{parquet_file}"
)

print(
    f"✅ Nome arquivo: "
    f"{parquet_file.name}"
)

print(
    f"✅ Total de registros: "
    f"{len(df)}"
)

print(
    f"✅ Total de colunas: "
    f"{len(df.columns)}"
)

print(
    f"✅ Tamanho do arquivo: "
    f"{file_size_kb} KB"
)

print(
    f"✅ Diretório temporário: "
    f"{temp_dir}"
)

✅ Parquet criado: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/jobs/tmp/2026-08-26_ibge_municipios.parquet
✅ Nome arquivo: 2026-08-26_ibge_municipios.parquet
✅ Total de registros: 5571
✅ Total de colunas: 24
✅ Tamanho do arquivo: 174.33 KB
✅ Diretório temporário: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/jobs/tmp


In [17]:
# =========================
# Upload Bronze
# =========================

# ------------------------------------------
# Validações
# ------------------------------------------

if df.empty:
    raise ValueError(
        "❌ DataFrame vazio. Nada para enviar."
    )

if not parquet_file.exists():
    raise FileNotFoundError(
        f"❌ Arquivo não encontrado: {parquet_file}"
    )

if not BRONZE_CONTAINER:
    raise ValueError(
        "❌ Container Bronze não configurado."
    )

if not blob_service_client:
    raise ValueError(
        "❌ Cliente Azure Blob não inicializado."
    )

# ------------------------------------------
# Nome do blob
# ------------------------------------------

blob_name = (
    f"{date_suffix}_ibge_municipios.parquet"
)

# ------------------------------------------
# Cliente Blob
# ------------------------------------------

blob_client = (
    blob_service_client.get_blob_client(
        container=BRONZE_CONTAINER,
        blob=blob_name
    )
)

# ------------------------------------------
# Upload
# ------------------------------------------

with open(
    parquet_file,
    "rb"
) as file_data:

    blob_client.upload_blob(
        file_data,
        overwrite=True
    )

# ------------------------------------------
# Validação Upload
# ------------------------------------------

if not blob_client.exists():
    raise RuntimeError(
        f"❌ Upload não encontrado no container: {blob_name}"
    )

blob_properties = (
    blob_client.get_blob_properties()
)

# ------------------------------------------
# Logs
# ------------------------------------------

print(
    f"✅ Upload concluído: "
    f"{blob_name}"
)

print(
    f"✅ Container: "
    f"{BRONZE_CONTAINER}"
)

print(
    f"✅ Arquivo local: "
    f"{parquet_file}"
)

print(
    f"✅ Registros enviados: "
    f"{len(df)}"
)

print(
    f"✅ Tamanho enviado: "
    f"{round(blob_properties.size / 1024, 2)} KB"
)

print(
    f"✅ Blob validado no Azure Storage"
)

✅ Upload concluído: 2026-08-26_ibge_municipios.parquet
✅ Container: bronze
✅ Arquivo local: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/jobs/tmp/2026-08-26_ibge_municipios.parquet
✅ Registros enviados: 5571
✅ Tamanho enviado: 174.33 KB
✅ Blob validado no Azure Storage
